<a href="https://colab.research.google.com/github/iaZe/data-analysis/blob/main/Spotify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 973.8 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488491 sha256=b65792df4c0385b7fd66137df1f1173e5ed789bc86125a73e485c5bdd9cea17a
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import pandas as pd

spark = SparkSession.builder.appName("Spotyfinho").getOrCreate()

data_path = "/content/spotify.csv"
dataframe = spark.read.csv(data_path, header = True, inferSchema = True)

dataframe.show()

+----------+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|Unnamed: 0|            track_id|             artists|          album_name|          track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|track_genre|
+----------+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|         0|5SuOikwiRyPMVoIQD...|         Gen Hoshino|              Comedy|              Comedy|        73|     230666|   False|       0.676| 0.461|  1|  -6.746|   0|      0.143|      0.0322|         1.01E-6|   0.358|

In [ ]:
dataframe = dataframe.dropna(subset=["track_name", "album_name"])
dataframe = dataframe.drop_duplicates(subset=["track_name", "album_name"])

dataframe.filter(col("popularity").cast("double").isNotNull())

dataframe.select("track_name", "artists", "album_name", "duration_ms", "popularity", "track_genre")
dataframe.show()

+--------------------+--------------------+--------------------------------+-----------+----------+-----------+
|          track_name|             artists|                      album_name|duration_ms|popularity|track_genre|
+--------------------+--------------------+--------------------------------+-----------+----------+-----------+
|"""A"" You're Ado...|        Brian Hyland|               The Bashful Blond|     151680|        39| rockabilly|
|"""Contemplate Th...|      Dillinger Four|                 C I V I L W A R|     180706|        24|  power-pop|
|"""DEVILS NEVER C...|   Capcom Sound Team|デビル メイ クライ 3 オリジナ...|     319906|        55|      anime|
|"""Eugene Onegin"...|     Nikolay Kopylov|             Popular Opera Arias|     111800|         0|    romance|
|"""Farewell"" Jin...|        Dave Brubeck|            A Dave Brubeck Ch...|     179400|        29|      piano|
|"""Gladiator"" - ...|Hans Zimmer;Lisa ...|            Hans Zimmer: Epic...|     313080|        16|     german|
|   

In [ ]:
dataframe.createOrReplaceTempView("view")

Mostrar o TOP 10 do spotify

In [ ]:
most_popular_music = spark.sql("SELECT track_name, artists, popularity FROM view ORDER BY popularity DESC LIMIT 10")
most_popular_music.show()

+--------------------+--------------------+----------+
|          track_name|             artists|popularity|
+--------------------+--------------------+----------+
|Quevedo: Bzrp Mus...|    Bizarrap;Quevedo|        99|
|     I'm Good (Blue)|David Guetta;Bebe...|        98|
|          La Bachata|       Manuel Turizo|        98|
|    Tití Me Preguntó|           Bad Bunny|        97|
|     Me Porto Bonito|Bad Bunny;Chencho...|        97|
|              Efecto|           Bad Bunny|        96|
|     I Ain't Worried|         OneRepublic|        96|
|       Ojitos Lindos|Bad Bunny;Bomba E...|        95|
|         Moscow Mule|           Bad Bunny|        94|
|             CUFF IT|             Beyoncé|        93|
+--------------------+--------------------+----------+



Mostrar a música mais famosa do spotify

In [ ]:
most_popular_album = spark.sql("SELECT * FROM view WHERE album_name = (SELECT album_name FROM view ORDER BY popularity DESC LIMIT 1)")
most_popular_album.show()

+--------------------+----------------+--------------------+-----------+----------+-----------+
|          track_name|         artists|          album_name|duration_ms|popularity|track_genre|
+--------------------+----------------+--------------------+-----------+----------+-----------+
|Quevedo: Bzrp Mus...|Bizarrap;Quevedo|Quevedo: Bzrp Mus...|     198937|        99|    hip-hop|
+--------------------+----------------+--------------------+-----------+----------+-----------+



Mostrar os albuns de um(a) cantor(a)

In [ ]:
singer = "Billie Eilish"
filter_singer = dataframe.select("artists", "album_name", "track_name").filter(dataframe.artists == singer)
filter_singer.show()

+-------------+--------------------+--------------------+
|      artists|          album_name|          track_name|
+-------------+--------------------+--------------------+
|Billie Eilish|   Happier Than Ever|   Billie Bossa Nova|
|Billie Eilish|               Bored|               Bored|
|Billie Eilish|    dont smile at me|             COPYCAT|
|Billie Eilish|   Happier Than Ever|I Didn't Change M...|
|Billie Eilish|   Halloween & Chill|          Lost Cause|
|Billie Eilish|   Happier Than Ever|                 NDA|
|Billie Eilish|   Happier Than Ever|            Oxytocin|
|Billie Eilish|Mega Hits Autumn/...|                  TV|
|Billie Eilish|Twisted Halloween...|all the good girl...|
|Billie Eilish|           Bellyache|           bellyache|
|Billie Eilish|bitches broken he...|bitches broken he...|
|Billie Eilish|Halloween Songs |...|       bury a friend|
|Billie Eilish|    dont smile at me|idontwannabeyouan...|
|Billie Eilish|    dont smile at me|              my boy|
|Billie Eilish

Mostrar a média de tempo de música dos artistas mais ouvidos

In [ ]:
average_time_popular_songs = spark.sql("SELECT artists, AVG(duration_ms) FROM view WHERE popularity >= 90 GROUP BY artists ORDER BY AVG(duration_ms) DESC LIMIT 10")
average_time_popular_songs.show()

+--------------------+----------------+
|             artists|avg(duration_ms)|
+--------------------+----------------+
|           Kate Bush|        298933.0|
|Bad Bunny;Bomba E...|        258298.0|
|           Tom Odell|        244360.0|
|The Weeknd;Gesaff...|        241066.0|
|   The Neighbourhood|        240400.0|
|   Rema;Selena Gomez|        239317.0|
|    Bad Bunny;Jhayco|        237894.0|
|             Ruth B.|        233720.0|
|The Weeknd;Daft Punk|        230453.0|
|Bad Bunny;Rauw Al...|        227628.0|
+--------------------+----------------+



Mostrar os ritmos mais populares

In [ ]:
dataframe_genre = dataframe.select("track_genre").distinct().orderBy(col("popularity").asc()).show()
dataframe_genre

+-------------+
|  track_genre|
+-------------+
|     acoustic|
|  alternative|
|      british|
|       disney|
|      romance|
|     hardcore|
|    hardstyle|
|        house|
|      new-age|
|      country|
|         jazz|
|    punk-rock|
|      turkish|
|    grindcore|
|    classical|
|drum-and-bass|
|       german|
|  black-metal|
|     cantopop|
|        tango|
+-------------+
only showing top 20 rows

